# Airbnb London: Gold Layer Exploratory Data Analysis & Statistical Testing

This notebook connects directly to our BigQuery `airbnb_gold` dataset to perform EDA and formal statistical testing on the transformed data.

## 1. Environment Setup & BigQuery Connection

In [ ]:
from google.cloud import bigquery
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Configure visual settings
sns.set_theme(style='whitegrid')

# Initialize BigQuery Client
# Note: This assumes you have run `gcloud auth application-default login`
client = bigquery.Client(project='expernetic-airbnb-pipeline')
print('BigQuery Client initialized successfully.')

## 2. Load Data from Gold Layer
Let's pull our enriched listing data from the `dim_listings` table.

In [ ]:
query_listings = """
    SELECT 
        listing_id,
        neighbourhood,
        room_type,
        price,
        total_reviews,
        avg_price_next_30_days
    FROM `expernetic-airbnb-pipeline.airbnb_gold.dim_listings`
    WHERE price > 0 -- filter out erroneous zero prices
"""

# Note: you may need to adjust the column names depending on your exact schema
df_listings = client.query(query_listings).to_dataframe()
df_listings.head()

## 3. Exploratory Data Analysis (EDA)
Let's look at the distribution of prices across different room types.

In [ ]:
plt.figure(figsize=(10, 6))
# Filtering out extreme outliers for better visualization
plot_data = df_listings[df_listings['price'] < 500]

sns.boxplot(x='room_type', y='price', data=plot_data)
plt.title('Price Distribution by Room Type (Filtered < $500)')
plt.ylabel('Price per Night (USD)')
plt.xlabel('Room Type')
plt.show()

## 4. Statistical Analysis: ANOVA Test
**Question:** Is there a statistically significant difference in pricing across different Room Types?

We will use a one-way ANOVA test to determine if the means are statistically different.

In [ ]:
# Prepare groups for ANOVA
groups = [group['price'].values for name, group in df_listings.groupby('room_type')]

# Perform One-Way ANOVA
f_stat, p_value = stats.f_oneway(*groups)

print(f'F-Statistic: {f_stat:.4f}')
print(f'P-Value: {p_value:.4e}')

if p_value < 0.05:
    print('\nConclusion: Since p < 0.05, we reject the null hypothesis. There is a statistically significant difference in average price across different room types.')
else:
    print('\nConclusion: Since p >= 0.05, we fail to reject the null hypothesis. We do not have enough evidence to say prices differ across room types.')

## 5. Statistical Analysis: Correlation
**Question:** Does the number of reviews correlate with the price of the listing?

In [ ]:
correlation, p_val = stats.pearsonr(df_listings['total_reviews'].fillna(0), df_listings['price'])

print(f'Pearson Correlation Coefficient: {correlation:.4f}')
print(f'P-Value: {p_val:.4e}')